In [1]:

# Idée du boosting : on part d'une prédiction simple, puis on ajoute
# des petits arbres L'UN APRÈS L'AUTRE. Chaque nouvel arbre apprend
# les erreurs laissées par les précédents.


ETA = 0.5        # η : pas d'apprentissage (on n'applique que la moitié de chaque correction)
LAMBDA = 0       # λ : régularisation des feuilles (0 = pas de régularisation)
N_ARBRES = 3     # nombre d'arbres enchaînés



#  Petits outils mathématiques écrits à la main

E = 2.718281828459045          # le nombre e

def exp(z):
    return E ** z

def ln(x):
    """Logarithme népérien par la méthode de Newton : on cherche y tel que e^y = x."""
    y = 0.0
    for _ in range(60):
        y = y - 1 + x / exp(y)
    return y

def sigmoide(F):
    """Transforme un score F en probabilité entre 0 et 1."""
    return 1 / (1 + exp(-F))

def moyenne(valeurs):
    return sum(valeurs) / len(valeurs)


#  UN PETIT ARBRE (une seule question « x <= seuil ? »)
#    Pour chaque exemple i, on a :
#      g_i = l'erreur à corriger        (y − F en régression, y − p en classification)
#      h_i = le « poids » de l'exemple  (1 en régression, p(1 − p) en classification)
#    Valeur d'une feuille = Σ g / (Σ h + λ)
#    Score d'un groupe    = (Σ g)² / (Σ h + λ)   (plus il est grand, mieux le groupe est corrigé)
#    Gain d'une question  = score(gauche) + score(droite) − score(tout le monde)
# ---------------------------------------------------------------------
def score(g, h, lam):
    return sum(g) ** 2 / (sum(h) + lam)

def valeur_feuille(g, h, lam):
    return sum(g) / (sum(h) + lam)

def apprendre_petit_arbre(X, g, h, lam, verbose=True):
    xs = sorted(set(X))
    seuils = [(a + b) / 2 for a, b in zip(xs, xs[1:])]     # milieux entre valeurs voisines
    score_tout = score(g, h, lam)
    meilleur = None
    for s in seuils:
        gauche = [i for i in range(len(X)) if X[i] <= s]   # positions des exemples à gauche
        droite = [i for i in range(len(X)) if X[i] > s]
        gain = (score([g[i] for i in gauche], [h[i] for i in gauche], lam)
                + score([g[i] for i in droite], [h[i] for i in droite], lam)
                - score_tout)
        if verbose:
            print(f"    question x <= {s:g} ?  gain = {gain:.3f}")
        if meilleur is None or gain > meilleur["gain"]:
            meilleur = {
                "seuil": s,
                "gain": gain,
                "gauche": valeur_feuille([g[i] for i in gauche], [h[i] for i in gauche], lam),
                "droite": valeur_feuille([g[i] for i in droite], [h[i] for i in droite], lam),
            }
    if verbose:
        print(f"  -> question gardée : x <= {meilleur['seuil']:g} ; "
              f"feuille gauche = {meilleur['gauche']:.2f} ; feuille droite = {meilleur['droite']:.2f}")
    return meilleur

def correction(arbre, x):
    """Ce que l'arbre propose d'ajouter pour un exemple x."""
    if x <= arbre["seuil"]:
        return arbre["gauche"]
    return arbre["droite"]

def arrondi(liste, n=2):
    return [round(v, n) for v in liste]


# RÉGRESSION : ventes de glaces (TND) selon la température (°C)

T = [10, 20, 30, 40]
VENTES = [10, 20, 60, 70]

def entrainer_regression(X, y, n_arbres, eta, lam, verbose=True):
    F0 = moyenne(y)                              # étape 1 : départ = la moyenne
    F = [F0] * len(y)                            # même prédiction pour tout le monde
    arbres = []
    mse = [moyenne([(y[i] - F[i]) ** 2 for i in range(len(y))])]
    if verbose:
        print(f"F0 = moyenne = {F0:g}  -> prédictions {arrondi(F)} ; MSE = {mse[0]:.1f}")
    for k in range(1, n_arbres + 1):
        g = [y[i] - F[i] for i in range(len(y))]     # étape 2 : les erreurs (résidus)
        h = [1] * len(y)                             # en régression, chaque exemple compte 1
        if verbose:
            print(f"\n--- ARBRE {k} --- erreurs à corriger : {arrondi(g)}")
        arbre = apprendre_petit_arbre(X, g, h, lam, verbose)   # étape 3 : arbre sur les erreurs
        arbres.append(arbre)
        F = [F[i] + eta * correction(arbre, X[i]) for i in range(len(y))]   # étape 4 : mise à jour
        mse.append(moyenne([(y[i] - F[i]) ** 2 for i in range(len(y))]))
        if verbose:
            print(f"  F{k} = F{k-1} + {eta} × feuille  -> prédictions {arrondi(F)} ; MSE = {mse[-1]:.1f}")
    return F0, arbres, mse

def predire_regression(F0, arbres, eta, x):
    F = F0
    for a in arbres:
        F += eta * correction(a, x)
    return F

print("=" * 65)
print("A. RÉGRESSION : ventes de glaces")
print("=" * 65)
F0, arbres_reg, mse = entrainer_regression(T, VENTES, N_ARBRES, ETA, LAMBDA)
print(f"\nMSE après 0, 1, 2, 3 arbres : {[round(m, 1) for m in mse]}")
x_nouveau = 35
print(f"Prédiction pour {x_nouveau} °C : {predire_regression(F0, arbres_reg, ETA, x_nouveau):.2f} TND")



#  CLASSIFICATION : l'étudiant réussit-il (1) ou échoue-t-il (0) ?
#    On travaille avec un score F (« log-odds »), puis p = σ(F).

HEURES = [1, 2, 3, 4]
REUSSITE = [0, 0, 1, 1]

def log_loss(y, p):
    total = 0
    for yi, pi in zip(y, p):
        total += -ln(pi) if yi == 1 else -ln(1 - pi)
    return total / len(y)

def entrainer_classification(X, y, n_arbres, eta, lam, verbose=True):
    F = [0.0] * len(y)                           # F0 = 0  ->  p = 0,5 pour tout le monde
    p = [sigmoide(f) for f in F]
    arbres = []
    pertes = [log_loss(y, p)]
    if verbose:
        print(f"F0 = 0 -> p = {arrondi(p)} ; perte = {pertes[0]:.2f}")
    for k in range(1, n_arbres + 1):
        g = [y[i] - p[i] for i in range(len(y))]         # erreur de probabilité y − p
        h = [p[i] * (1 - p[i]) for i in range(len(y))]   # « courbure » p(1 − p)
        if verbose:
            print(f"\n--- ARBRE {k} --- erreurs y − p = {arrondi(g)} ; p(1 − p) = {arrondi(h)}")
        arbre = apprendre_petit_arbre(X, g, h, lam, verbose)
        arbres.append(arbre)
        F = [F[i] + eta * correction(arbre, X[i]) for i in range(len(y))]
        p = [sigmoide(f) for f in F]
        pertes.append(log_loss(y, p))
        if verbose:
            print(f"  F{k} = {arrondi(F)} -> p = {arrondi(p)} ; perte = {pertes[-1]:.2f}")
    return arbres, pertes

def predire_proba(arbres, eta, x):
    F = 0.0
    for a in arbres:
        F += eta * correction(a, x)
    return F, sigmoide(F)

print("\n" + "=" * 65)
print("B. CLASSIFICATION : réussite des étudiants")
print("=" * 65)
arbres_clf, pertes = entrainer_classification(HEURES, REUSSITE, N_ARBRES, ETA, LAMBDA)
print(f"\nPerte après 0, 1, 2, 3 arbres : {arrondi(pertes)}")
x_nouveau = 3.5
F, p = predire_proba(arbres_clf, ETA, x_nouveau)
print(f"Étudiant de {x_nouveau} h : F = {F:.2f} -> p = {p:.2f} -> "
      f"{'RÉUSSITE' if p >= 0.5 else 'ÉCHEC'}")



# C. EFFET DE λ : même exemple des glaces, trois valeurs de λ
#    λ grand -> corrections plus petites -> modèle plus prudent

print("\n" + "=" * 65)
print("C. EFFET DE λ (ventes de glaces)")
print("=" * 65)
for lam in [0, 1, 10]:
    _, arbres_l, mse_l = entrainer_regression(T, VENTES, N_ARBRES, ETA, lam, verbose=False)
    print(f"λ = {lam:2d} : feuille gauche de l'arbre 1 = {arbres_l[0]['gauche']:6.2f} ; "
          f"MSE après 0..3 arbres = {[round(m) for m in mse_l]}")

A. RÉGRESSION : ventes de glaces
F0 = moyenne = 40  -> prédictions [40.0, 40.0, 40.0, 40.0] ; MSE = 650.0

--- ARBRE 1 --- erreurs à corriger : [-30.0, -20.0, 20.0, 30.0]
    question x <= 15 ?  gain = 1200.000
    question x <= 25 ?  gain = 2500.000
    question x <= 35 ?  gain = 1200.000
  -> question gardée : x <= 25 ; feuille gauche = -25.00 ; feuille droite = 25.00
  F1 = F0 + 0.5 × feuille  -> prédictions [27.5, 27.5, 52.5, 52.5] ; MSE = 181.2

--- ARBRE 2 --- erreurs à corriger : [-17.5, -7.5, 7.5, 17.5]
    question x <= 15 ?  gain = 408.333
    question x <= 25 ?  gain = 625.000
    question x <= 35 ?  gain = 408.333
  -> question gardée : x <= 25 ; feuille gauche = -12.50 ; feuille droite = 12.50
  F2 = F1 + 0.5 × feuille  -> prédictions [21.25, 21.25, 58.75, 58.75] ; MSE = 64.1

--- ARBRE 3 --- erreurs à corriger : [-11.25, -1.25, 1.25, 11.25]
    question x <= 15 ?  gain = 168.750
    question x <= 25 ?  gain = 156.250
    question x <= 35 ?  gain = 168.750
  -> question ga